# Setup

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
TRAJ_PATH = PROJECT_ROOT / "data" / "topic_trajectories.pkl"

print("TRAJ_PATH exists:", TRAJ_PATH.exists(), TRAJ_PATH)

TRAJ_PATH exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/data/topic_trajectories.pkl


In [2]:
# Load saved trajectories from pickle
with open(TRAJ_PATH, "rb") as f:
    topic_trajectories = pickle.load(f)

print("Number of topics loaded:", len(topic_trajectories))

for topic_id, info in list(topic_trajectories.items())[:3]:
    print(f"\nTopic {topic_id}:")
    print("Label:", info["label"])
    print("Years:", info["years"])
    print("Trajectory shape:", info["trajectory"].shape)

Number of topics loaded: 15

Topic 0:
Label: Public Health & Workforce Impact
Years: [2020, 2021, 2022]
Trajectory shape: (3, 384)

Topic 1:
Label: Online Education & Training
Years: [2020, 2021, 2022]
Trajectory shape: (3, 384)

Topic 2:
Label: Clinical Virology & Infection
Years: [2020, 2021, 2022]
Trajectory shape: (3, 384)


In [3]:
# build transition matrices
transition_rows = []

for topic_id, info in topic_trajectories.items():
    years = info["years"]
    traj = info["trajectory"]
    label = info["label"]
    
    for t in range(len(years) - 1):
        transition_rows.append({
            "topic_id": topic_id,
            "topic_label": label,
            "year_from": years[t],
            "year_to": years[t + 1],
            "x_t": traj[t],
            "x_t1": traj[t + 1],
        })

transitions_df = pd.DataFrame(transition_rows)

print("Transition dataset shape:", transitions_df.shape)
display(transitions_df.head())

Transition dataset shape: (27, 6)


,topic_id,topic_label,year_from,year_to,x_t,x_t1
0,0,Public Health & Workforce Impact,2020,2021,"[0.022808092, 0.03738438, -0.025412643, 0.0275...","[0.00894767, 0.015065819, -0.009621622, 0.0234..."
1,0,Public Health & Workforce Impact,2021,2022,"[0.00894767, 0.015065819, -0.009621622, 0.0234...","[0.036359653, 0.0104295965, 0.013622075, 0.005..."
2,1,Online Education & Training,2020,2021,"[0.01232472, 0.029829707, 0.0024571244, 0.0058...","[0.008245414, 0.015392465, 0.0030519473, -0.02..."
3,1,Online Education & Training,2021,2022,"[0.008245414, 0.015392465, 0.0030519473, -0.02...","[0.022293752, 0.006708989, -0.041292395, -0.02..."
4,2,Clinical Virology & Infection,2020,2021,"[0.010815447, 0.026126163, -0.027783157, 0.007...","[-0.007693753, 0.017500222, -0.029099848, 0.00..."


In [4]:
# stack
X_t = np.vstack(transitions_df["x_t"].values)
X_t1 = np.vstack(transitions_df["x_t1"].values)

print("X_t shape:", X_t.shape)
print("X_t1 shape:", X_t1.shape)

X_t shape: (27, 384)
X_t1 shape: (27, 384)


# Denoising

In [5]:
# Add Gaussian Noise
rng = np.random.default_rng(2020)

NOISE_STD = 0.05

X_t1_noisy = X_t1 + rng.normal(
    loc=0.0,
    scale=NOISE_STD,
    size=X_t1.shape
)

print("Noise std:", NOISE_STD)
print("X_t1_noisy shape:", X_t1_noisy.shape)

# quick sanity check
diff_norms = np.linalg.norm(X_t1_noisy - X_t1, axis=1)
print("Mean noise L2 norm:", diff_norms.mean())
print("Min/Max noise L2 norm:", diff_norms.min(), diff_norms.max())

Noise std: 0.05
X_t1_noisy shape: (27, 384)
Mean noise L2 norm: 0.9829302890265231
Min/Max noise L2 norm: 0.9065525519489105 1.0347625586158793


In [6]:
# Build Model Inputs
# Input = current clean state + noisy next state
X_input = np.hstack([X_t, X_t1_noisy])

# Target = clean next state
Y_target = X_t1.copy()

print("X_input shape:", X_input.shape)
print("Y_target shape:", Y_target.shape)

X_input shape: (27, 768)
Y_target shape: (27, 384)


In [7]:
# Ridge Reg Denoiser
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_input, Y_target)

Y_pred = ridge.predict(X_input)

print("Predicted shape:", Y_pred.shape)

Predicted shape: (27, 384)


In [8]:
# Quality check
mse_noisy = np.mean((X_t1_noisy - X_t1) ** 2)
mse_pred = np.mean((Y_pred - X_t1) ** 2)

print("MSE of noisy next-state vs clean next-state:", mse_noisy)
print("MSE of predicted next-state vs clean next-state:", mse_pred)
print("Improvement:", mse_noisy - mse_pred)

MSE of noisy next-state vs clean next-state: 0.0025192809436176693
MSE of predicted next-state vs clean next-state: 0.00011423406483890995
Improvement: 0.0024050468787787595


# Simulate the topic over time

In [9]:
# Get latest states
latest_states = []

for topic_id, info in topic_trajectories.items():
    years = info["years"]
    traj = info["trajectory"]
    
    if years[-1] == 2022:
        latest_states.append({
            "topic_id": topic_id,
            "label": info["label"],
            "x_2022": traj[-1]
        })

latest_df = pd.DataFrame(latest_states)

X_2022 = np.vstack(latest_df["x_2022"].values)

print("Latest states shape:", X_2022.shape)
display(latest_df.head())

Latest states shape: (12, 384)


,topic_id,label,x_2022
0,0,Public Health & Workforce Impact,"[0.036359653, 0.0104295965, 0.013622075, 0.005..."
1,1,Online Education & Training,"[0.022293752, 0.006708989, -0.041292395, -0.02..."
2,2,Clinical Virology & Infection,"[-0.012340484, 0.01592598, -0.019865632, 0.004..."
3,3,Medical Imaging & Deep Learning,"[-0.0033894896, 0.020917473, 0.030582165, -0.0..."
4,4,Financial Markets & Economic Impact,"[-0.057611395, -0.09756207, 0.04446043, 0.1054..."


In [10]:
# Simulate noisy data
NOISE_STD_FUTURE = 0.05

X_2023_noisy = X_2022 + rng.normal(
    0.0,
    NOISE_STD_FUTURE,
    X_2022.shape
)

print("Simulated noisy future shape:", X_2023_noisy.shape)

Simulated noisy future shape: (12, 384)


In [11]:
# Denoise
X_input_future = np.hstack([X_2022, X_2023_noisy])

X_2023_pred = ridge.predict(X_input_future)

print("Predicted future shape:", X_2023_pred.shape)

Predicted future shape: (12, 384)


In [12]:
# inspect movement magnitude
movement = np.linalg.norm(X_2023_pred - X_2022, axis=1)

results_df = latest_df.copy()
results_df["movement_norm"] = movement

display(results_df.sort_values("movement_norm", ascending=False))

,topic_id,label,x_2022,movement_norm
4,4,Financial Markets & Economic Impact,"[-0.057611395, -0.09756207, 0.04446043, 0.1054...",0.547584
8,10,"Prevention, Masks & Environmental Effects","[0.037953418, -0.047686562, -0.018264616, -0.0...",0.536653
7,9,Food Systems & Hospitality Impact,"[0.007989797, 0.015353438, 0.030304627, 0.0889...",0.534221
9,11,"Policy, Governance & Public Response","[-0.060008418, -0.0053242818, -0.008966697, -0...",0.511436
2,2,Clinical Virology & Infection,"[-0.012340484, 0.01592598, -0.019865632, 0.004...",0.496899
6,8,Severe Disease & Immune Response,"[-0.0431109, -0.0029038899, -0.026397713, 0.02...",0.496475
3,3,Medical Imaging & Deep Learning,"[-0.0033894896, 0.020917473, 0.030582165, -0.0...",0.472628
0,0,Public Health & Workforce Impact,"[0.036359653, 0.0104295965, 0.013622075, 0.005...",0.453583
1,1,Online Education & Training,"[0.022293752, 0.006708989, -0.041292395, -0.02...",0.451147
5,5,Mental Health & Psychological Effects,"[0.04922828, 0.0743325, -0.0013035903, 0.08264...",0.428967
